# Two Moons — Network Selection HPO

This notebook shows how to use `bayesflow_hpo` to **automatically compare different inference network architectures** on the classic [Two Moons](https://bayesflow.org/main/_examples/Two_Moons_Starter.html) benchmark.

Instead of manually training each network type (as in the original example), we set up a single Optuna study with a `NetworkSelectionSpace` that searches across **Flow Matching**, **Coupling Flow**, and **Consistency Model** simultaneously — optimizing both the network choice and its hyperparameters.

See the [getting started notebook](getting_started.ipynb) for a general introduction to `bayesflow_hpo`.

In [ ]:
%pip install --quiet --upgrade -e ..

In [ ]:
import numpy as np
import bayesflow as bf
import bayesflow_hpo as hpo

## 1. Simulator & Adapter

The Two Moons model has a 2-D uniform prior and a stochastic forward model that produces a characteristic crescent-shaped posterior. See the [original example](https://bayesflow.org/main/_examples/Two_Moons_Starter.html) for details.

Since the observations `x` are fixed-size vectors (not sets), we use them directly as `inference_conditions` — no summary network is needed.

In [ ]:
def prior():
    theta = np.random.uniform(-1, 1, 2).astype("float32")
    return dict(theta=theta)


def forward_model(theta):
    alpha = np.random.uniform(-np.pi / 2, np.pi / 2)
    r = np.random.normal(0.1, 0.01)
    x1 = -np.abs(theta[0] + theta[1]) / np.sqrt(2) + r * np.cos(alpha) + 0.25
    x2 = (-theta[0] + theta[1]) / np.sqrt(2) + r * np.sin(alpha)
    return dict(x=np.array([x1, x2], dtype="float32"))


simulator = bf.make_simulator([prior, forward_model])
adapter = (
    bf.Adapter()
    .rename("theta", "inference_variables")
    .rename("x", "inference_conditions")
)

## 2. Network Selection Search Space

The key ingredient: `NetworkSelectionSpace` wraps multiple inference network spaces and adds a categorical Optuna parameter (`inference_network_type`) that selects which one to use in each trial. Optuna then jointly optimizes the network choice and its architecture hyperparameters.

In [ ]:
search_space = hpo.CompositeSearchSpace(
    inference_space=hpo.NetworkSelectionSpace(
        candidates={
            "flow_matching": hpo.FlowMatchingSpace(),
            "coupling_flow": hpo.CouplingFlowSpace(),
            "consistency_model": hpo.ConsistencyModelSpace(),
        }
    ),
    summary_space=None,
    training_space=hpo.TrainingSpace(),
)

## 3. Run HPO

We run a small study (8 trials) to compare the three network types. In practice, increase `n_trials` for more thorough exploration.

In [ ]:
def train_fn(approximator, simulator, hparams, callbacks):
    """Compatibility hook for BayesFlow 2.0.8+ (maps batches_per_epoch → num_batches)."""
    approximator.fit(
        simulator=simulator,
        epochs=int(hparams["epochs"]),
        batch_size=int(hparams.get("batch_size", 256)),
        num_batches=int(hparams["batches_per_epoch"]),
        callbacks=callbacks,
    )


study = hpo.optimize(
    simulator=simulator,
    adapter=adapter,
    search_space=search_space,
    n_trials=8,
    epochs=30,
    batches_per_epoch=50,
    max_param_count=500_000,
    objective_metrics=["calibration_error", "nrmse"],
    objective_mode="pareto",
    train_fn=train_fn,
    storage=None,
    show_progress_bar=False,
)

## 4. Results

In [ ]:
print(hpo.summarize_study(study))

In [ ]:
fig = hpo.plot_study(study)

### Trial Table

The `inference_network_type` column shows which network Optuna selected for each trial, making it easy to compare performance across architectures.

In [ ]:
df = hpo.trials_to_dataframe(study)

key_cols = [
    "trial_number", "inference_network_type",
    "calibration_error", "nrmse", "correlation",
    "coverage_90", "param_count", "inference_time_s",
]
display_cols = [c for c in key_cols if c in df.columns]
df.sort_values("calibration_error")[display_cols]

### Best Configuration

In [ ]:
table = hpo.trial_table(study, top_k=5, metrics=["correlation", "coverage_90"])
table

In [ ]:
config = hpo.best_config(study)
config

## 5. Retrain Best Model

Rebuild the winning architecture from the HPO config and retrain with a larger budget.

In [ ]:
import keras

approximator = hpo.build_continuous_approximator(config, adapter, search_space)

epochs, batches_per_epoch = 75, 100
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=float(config["initial_lr"]),
    decay_steps=epochs * batches_per_epoch,
)
approximator.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule))

approximator.fit(
    simulator=simulator,
    epochs=epochs,
    num_batches=batches_per_epoch,
    batch_size=int(config.get("batch_size", 256)),
)

### Posterior Samples

Draw posterior samples at the canonical observation $x = (0, 0)$ to verify the characteristic crescent shape.

In [ ]:
import matplotlib.pyplot as plt

conditions = {"x": np.array([[0.0, 0.0]], dtype="float32")}
samples = approximator.sample(conditions=conditions, num_samples=3000)["theta"]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(samples[0, :, 0], samples[0, :, 1], alpha=0.5, s=0.5)
ax.set(xlabel=r"$\theta_1$", ylabel=r"$\theta_2$", xlim=(-0.5, 0.5), ylim=(-0.5, 0.5))
ax.set_aspect("equal")
ax.set_title(f"Best network: {config.get('inference_network_type', 'N/A')}")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()